# Day 01 — Data Cleaning Challenge
**Dataset:** Synthetic Manufacturing Quality Data (Factory-style: Cpk, Ppk, Dosing Accuracy)

**Goal:** Identify and fix all data quality issues → produce a clean, analysis-ready dataset.


## Step 1: Load & Scope the Mess

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_factory_data.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
# Dtypes — notice Cp, Cpk, Dosing_Accuracy_% are 'object' instead of float
df.info()

In [ ]:
# Missing values
df.isnull().sum()

In [ ]:
# Duplicate rows
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Inconsistent Factory names
df['Factory'].value_counts()

In [ ]:
# Mixed date formats
df['Date'].unique()[:20]

## Step 2: Fix Duplicates

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

## Step 3: Fix Data Types (Cp, Cpk, Dosing_Accuracy_%)
These columns contain string junk like `'N/A'`, `'-'`, `'error'` mixed with numeric values.

In [ ]:
# See what the non-numeric values look like
for col in ['Cp', 'Cpk', 'Dosing_Accuracy_%']:
    non_numeric = df[col].apply(lambda x: pd.to_numeric(x, errors='coerce')).isna() & df[col].notna()
    print(f'{col} dirty values:', df[col][non_numeric].unique())

In [ ]:
# Convert: string junk becomes NaN, then we treat them like missing values
for col in ['Cp', 'Cpk', 'Dosing_Accuracy_%']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes

## Step 4: Handle Missing Values

In [ ]:
print('Missing values after type fix:')
print(df.isnull().sum())

In [ ]:
# Strategy: impute numeric quality metrics with column median (robust to outliers)
# Why median not mean? Because we injected some extreme outliers in Cpk and Dosing.
for col in ['Cp', 'Cpk', 'Pp', 'Ppk', 'Dosing_Accuracy_%']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'{col} median used for imputation: {median_val:.3f}')

In [ ]:
# Confirm no more nulls
df.isnull().sum()

## Step 5: Standardize Factory Names

In [ ]:
factory_map = {
    'pondicherry pc': 'Pondicherry PC',
    'pondichery pc':  'Pondicherry PC',
    'pondicherry':    'Pondicherry PC',
    'llpl':           'LLPL',
    'l.l.p.l':        'LLPL',
    'llpl ':          'LLPL',
    'llpl':           'LLPL',
    'nepal pc':       'Nepal PC',
    'nepal pc':       'Nepal PC',
    'nepal':          'Nepal PC',
    'haridwar pc':    'Haridwar PC',
    'haridwar':       'Haridwar PC',
    'khamgaon':       'Khamgaon',
    'khamagon':       'Khamgaon',
    'khamgoan':       'Khamgaon',
    'sumerpur pc':    'Sumerpur PC',
    'sumerpur':       'Sumerpur PC',
}

df['Factory'] = df['Factory'].str.strip().str.lower().map(
    lambda x: factory_map.get(x, x.title())
)

df['Factory'].value_counts()

## Step 6: Standardize Dates

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=False, infer_datetime_format=True, errors='coerce')
print('Date nulls after parsing:', df['Date'].isna().sum())
df['Date'].dtype

## Step 7: Detect & Flag Outliers

In [ ]:
# IQR method on Cpk and Dosing_Accuracy_%
for col in ['Cpk', 'Dosing_Accuracy_%']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f'{col}: {len(outliers)} outliers detected')
    print(outliers[[col]].describe())

In [ ]:
# Flag them — don't drop, just mark for downstream analysis
for col in ['Cpk', 'Dosing_Accuracy_%']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[f'{col}_outlier'] = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)

## Step 8: Final Clean Dataset

In [ ]:
print('Final shape:', df.shape)
df.info()
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
df.to_csv('clean_factory_data.csv', index=False)
print('Saved clean_factory_data.csv')

## Summary

| Issue Found | Fix Applied |
|---|---|
| 15 duplicate rows | Dropped with `drop_duplicates()` |
| Cp, Cpk, Dosing had string junk (`N/A`, `-`, `error`) | `pd.to_numeric(errors='coerce')` → treated as NaN |
| Missing values in all numeric columns | Imputed with column median (robust to outliers) |
| 30+ variants of factory names (inconsistent case, spelling) | Standardized via `.str.lower()` + mapping dictionary |
| 4 different date formats | Parsed to datetime with `pd.to_datetime(infer_datetime_format=True)` |
| Outliers in Cpk and Dosing | Detected with IQR method, flagged (not dropped) |

**Before:** 315 rows, 9 columns, multiple dtype issues, dirty values, inconsistent text  
**After:** 300 rows, 11 columns (2 outlier flags added), fully typed, standardized, analysis-ready
